# FedCardioTwin: J-BHI revision runs (Kaggle, 2 x T4)

**Before running:** Settings: Accelerator **GPU T4 x2**, **Internet on**. Add Input: `hammad0110/ieee-2` (cache) and `hammad0110/chapman` (fixed Chapman X.npy).

**How to run a batch unattended:** set `BATCH` in cell 1, then **Save Version → Save & Run All (Commit)**. When it finishes, open the version's **Output** tab and download `out_<BATCH>.zip`.

| Batch | GPU 0 | GPU 1 | ~Wall time |
|---|---|---|---|
| A1 | Centralized, FedAvg, FedPer (120 rounds, 3 seeds), FedAvg+FT / Central+FT, FedPer ensemble, FedBABU seed 2 (+ profiling before training) | Local, FedRep, FedBABU seeds 0-1 (120 rounds) | ~6 h |
| A2 | Ditto, FedAvgM seed 2 (120 rounds) | FedProx, FedBN, FedAvgM seeds 0-1 (120 rounds) | ~5.5 h |
| B | reference FedAvg/FedPer (60 rounds), depth +1/+2, equal and FedLA weighting | partial participation, dropout, 6/3/2/1 leads | ~6.5 h |
| C | LOHO + onboarding, twin (Track B), federated twin, FedProx/Ditto sensitivity | GroupNorm baseline, DP-SGD eps 8/3/1 | ~6 h |
| C2 | FedProx/Ditto sensitivity | GroupNorm baseline, DP-SGD eps 8/3/1 (use this instead of C if C1 runs on the Mac) | ~5 h |

Batches are independent; run them in any order (one commit each). Every run skips work whose result file already exists.


In [ ]:
# 1. Choose the batch
BATCH = "A1"          # "A1" | "A2" | "B" | "C" | "C2"
SMOKE_FIRST = True    # quick 2-round GPU check of every new code path (~5 min)
GIT_BRANCH = "revision"
REPO_URL = "https://github.com/perfectenshclag/FedCardioTwin.git"
START_LIMIT_H = 9.5   # do not START new jobs after this many hours
KILL_LIMIT_H = 11.3   # stop running jobs and package outputs (Kaggle limit 12 h)


In [ ]:
# 2. Setup: code, packages, evaluation repo, writable cache
import os, sys, glob, shutil, subprocess, time, json
T0 = time.time()
W = "/kaggle/working"
REPO = f"{W}/FedCardioTwin"
os.chdir(W)

# code: an attached dataset holding the repo wins, else clone the branch
src = [os.path.dirname(os.path.dirname(p)) for p in
       glob.glob("/kaggle/input/**/fedcardiotwin/fl/engine.py", recursive=True)]
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if src:
    shutil.copytree(src[0], REPO)
    print("code from dataset:", src[0])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GIT_BRANCH,
                    REPO_URL, REPO], check=True)
    print("code from git:", GIT_BRANCH,
          subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"],
                         capture_output=True, text=True).stdout.strip())
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opacus==1.5.4"],
               check=False)
if not os.path.exists("external/evaluation-2021/weights.csv"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/physionetchallenges/evaluation-2021",
                    "external/evaluation-2021"], check=True)

# writable cache: symlink every client folder; for each client take the
# LARGEST X.npy found (the ieee-2 Chapman copy is truncated)
C = f"{W}/cache"
os.makedirs(C, exist_ok=True)
import numpy as np
CLIENTS = ["CPSC", "CPSC-Extra", "Georgia", "Chapman", "Ningbo", "PTB-XL", "PTBXL_TRACKB"]
allx = glob.glob("/kaggle/input/**/X.npy", recursive=True)
for c in CLIENTS:
    if c == "Chapman":
        cand = [p for p in allx if "hapman" in p]
    else:
        cand = [p for p in allx if os.path.basename(os.path.dirname(p)) == c]
    assert cand, f"no X.npy for {c}"
    xbest = max(cand, key=os.path.getsize)
    d = f"{C}/{c}"
    os.makedirs(d, exist_ok=True)
    for f in ("X.npy", "Y.npy", "manifest.csv", "meta.csv"):
        dst = f"{d}/{f}"
        if os.path.lexists(dst):
            os.remove(dst)
        srcf = os.path.join(os.path.dirname(xbest), f)
        if not os.path.exists(srcf):  # Y/manifest may live in the ieee-2 copy
            alt = [p for p in glob.glob(f"/kaggle/input/**/{c}/{f}", recursive=True)]
            srcf = alt[0] if alt else None
        if srcf:
            os.symlink(srcf, dst)
    X = np.load(f"{d}/X.npy", mmap_mode="r"); Y = np.load(f"{d}/Y.npy")
    assert len(X) == len(Y), (c, X.shape, Y.shape)
    print(f"{c:13s} X={X.shape} Y={Y.shape} <- {xbest}")
if not os.path.exists(f"{C}/ptbxl_database.csv"):
    subprocess.run(["curl", "-sSL", "-o", f"{C}/ptbxl_database.csv",
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"],
                   check=True)
print("ptbxl_database.csv:", os.path.getsize(f"{C}/ptbxl_database.csv"), "bytes")

import torch
print("torch", torch.__version__, "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
RES, CK, LOGS = f"{W}/results", f"{W}/checkpoints", f"{W}/logs"
for d in (RES, CK, LOGS):
    os.makedirs(d, exist_ok=True)


In [ ]:
# 3. Batch definitions (shared with scripts/run_batch.py)
sys.path.insert(0, f"{REPO}/scripts")
from batches import make_batches
BATCHES = make_batches("python", C, RES, CK)
for g, q in enumerate(BATCHES[BATCH]):
    print(f"GPU{g}:", [j[0] for j in q])


In [ ]:
# 4. Smoke check on GPU (2 rounds, small subsets) -- aborts the batch on failure
if SMOKE_FIRST:
    SR, SC = f"{W}/smoke_results", f"{W}/smoke_ckpt"
    SB = f"python -u scripts/run_experiments.py --cache-dir {C} --results-dir {SR} --ckpt-dir {SC} --preset fast --max-records 400 --seeds 0"
    smoke = {
        "A1": [f"{SB} --stage federated --strategies fedavg fedper fedrep fedbabu local --rounds 2 --main-round 1 --snapshot-rounds 1 2 --eval-every 1",
               f"{SB} --stage centralized", f"{SB} --stage finetune"],
        "A2": [f"{SB} --stage federated --strategies ditto fedprox fedbn fedavgm --rounds 2"],
        "B":  [f"{SB} --stage federated --strategies fedper --local-blocks 1 --weighting label --participation 0.5 --dropout 0.3 --rounds 2",
               f"{SB} --stage federated --strategies fedper --leads 1 --tag _l1 --rounds 2"],
        "C2": [f"{SB} --stage federated --strategies fedper --dp-epsilon 8 --tag _dp --rounds 1",
               f"{SB} --stage federated --strategies fedprox --prox-mu 0.1 --tag _hp --rounds 1"],
        "C":  [f"{SB} --stage federated --strategies fedper --dp-epsilon 8 --tag _dp --rounds 1",
               f"{SB} --stage federated --strategies fedper --twin-split --tag _twinfed --rounds 1",
               f"{SB} --stage twin_fed",
               f"{SB} --stage loho --hospitals CPSC-Extra --onboard-sizes 50 0 --rounds 1"],
    }[BATCH]
    for cmd in smoke:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                           env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
        print(("OK   " if r.returncode == 0 else "FAIL ") + cmd.split("--stage")[1][:90])
        if r.returncode != 0 or "FAILED" in r.stdout + r.stderr:
            print(r.stdout[-3000:], r.stderr[-3000:])
            raise RuntimeError("smoke check failed; fix before the full batch")
    shutil.rmtree(SR, ignore_errors=True); shutil.rmtree(SC, ignore_errors=True)
    print("smoke check passed in %.1f min" % ((time.time() - T0) / 60))

# measured compute footprint (idle machine, before the training queues start)
if BATCH == "A1" and not os.path.exists(f"{RES}/profile.json"):
    subprocess.run(f"python -u scripts/profile_compute.py --out {RES}/profile.json",
                   shell=True, check=False, cwd=REPO,
                   env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})


In [ ]:
# 5. Run the batch on both GPUs (each queue sequential; skips finished runs)
import threading
status = {}

def run_queue(gpu, jobs):
    for name, cmd, needs in jobs:
        if time.time() - T0 > START_LIMIT_H * 3600:
            status[name] = "not started (time)"; continue
        while needs and not all(os.path.exists(p) for p in needs):
            if time.time() - T0 > START_LIMIT_H * 3600:
                break
            time.sleep(60)
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        t = time.time()
        with open(f"{LOGS}/{name}.log", "a") as f:
            p = subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT,
                                 env=env, cwd=REPO)
            while p.poll() is None:
                if time.time() - T0 > KILL_LIMIT_H * 3600:
                    p.kill(); status[name] = "killed (time)"; break
                time.sleep(20)
        status.setdefault(name, f"exit {p.returncode} in {(time.time() - t) / 3600:.2f} h")
        print(f"[GPU{gpu}] {name}: {status[name]}", flush=True)

threads = [threading.Thread(target=run_queue, args=(g, q))
           for g, q in enumerate(BATCHES[BATCH])]
for th in threads: th.start()
while any(th.is_alive() for th in threads):
    time.sleep(600)
    done = sorted(os.path.basename(p) for p in glob.glob(f"{RES}/*.json"))
    print(f"[{(time.time() - T0) / 3600:.1f} h] {len(done)} result files", flush=True)
for th in threads: th.join()
print(json.dumps(status, indent=1))


In [ ]:
# 6. Package outputs -> /kaggle/working/out_<BATCH>.zip (download from the Output tab)
import zipfile
keep_ck = ("_clients.pt",)
keep_names = ("fed_fedavg_seed", "central_seed", "loho_", "twin_base", "fed_fedper_twinfed")
zpath = f"{W}/out_{BATCH}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root in (RES, LOGS):
        for p in glob.glob(f"{root}/**/*", recursive=True):
            if os.path.isfile(p):
                z.write(p, os.path.relpath(p, W))
    for p in glob.glob(f"{CK}/*.pt"):
        b = os.path.basename(p)
        if "_progress" in b:
            continue
        if b.endswith(keep_ck) or b.startswith(keep_names):
            z.write(p, os.path.relpath(p, W))
print(zpath, round(os.path.getsize(zpath) / 2**20, 1), "MiB")
print(json.dumps(status, indent=1))
